# C5 · Raster algebra and reclassification: elevation zones of the Kathmandu Valley

> **Time:** about 2 hours  
> **You need:** Parts A and B (especially A8 · NumPy arrays and A9 · Pandas tables), and Lessons C1 to C4, especially nodata (C1) and the UTM 45N grid with 30 m cells (C4).  
> **By the end you can:**
> 1. calculate with whole rasters at once (**map algebra**), and avoid the integer **overflow** trap by converting to float,
> 2. build masks with comparisons, combine them with `&`, `|` and `~`, and use `np.where` and `xr.where` to choose values cell by cell,
> 3. **reclassify** the DEM into zones, measure each zone's area in km², and handle nodata with `NaN`, masked arrays and `fillnodata`.

In Lesson A8 you added, compared and masked a small made-up grid of elevations. This lesson does the same with the real DEM: 3.3 million cells at once, with one line of code per idea. GIS calls this **map algebra** or **raster algebra**.

Every calculation runs on the grid from Lesson C4: the DEM reprojected to **UTM 45N with 30 m cells**, so that every cell covers the same 900 m² of ground and counting cells measures area.

*Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA).* The full licence notice is in `data/README.md`.

**Counting lines:** lines that only set up or show a plot (`fig, ax = plt.subplots()`, `plt.show()`) do not count toward the Q1/Q2/Q3 line counts.

**About the numbers:** resampling can differ very slightly between library versions, so a few counts may differ from the text in the last digits.

**How to practise:** after every step there are three questions. **Q1** needs one line of code, **Q2** two lines, **Q3** about three. Type your answer in the empty cell under each question, run it, then open **Show solution** to check. Try first, even if you are not sure: struggling for a few minutes is how the skill grows.

<!-- pygile-nav -->
**PyGILE Course** · lesson 29 of 39 · [Course home](../README.md)  
← Previous: [C4 · Reproject, resample and mosaic: the DEM on a grid in metres](C04_Reproject_resample_and_mosaic.ipynb) · Next: [C6 · Terrain analysis: slope, aspect and hillshade for landslide studies](C06_Terrain_analysis.ipynb) →

## 1. Arithmetic on whole grids

In NumPy (A8) and xarray, an arithmetic sign works on **every cell at once**: `dem_utm * 3.28084` converts all 3.3 million elevations to feet in one line. No loop is needed.

With xarray, the result is again a DataArray with the **same coordinates and CRS**. It is still a map, ready for `.rio` methods.

A classic map-algebra product is **relative elevation**: each cell minus the mean of the whole grid. Positive cells lie above the average ground, negative cells below. It needs one number, the mean, subtracted from every cell; NumPy repeats the single number for you (**broadcasting**, A8).

The first cell repeats the C4 recipe and builds `dem_utm`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
import rioxarray as rxr
import xarray as xr
from rasterio.enums import Resampling

DATA = Path("..") / "data"
OUT = Path("..") / "outputs"
OUT.mkdir(exist_ok=True)
DEM_PATH = DATA / "dem_kathmandu_melamchi.tif"
CELL_KM2 = 30 * 30 / 1000000    # one 30 m cell is 0.0009 km²

# As in C4: UTM 45N, 30 m cells, bilinear resampling; NaN where there is no data
dem = rxr.open_rasterio(DEM_PATH, masked=True).squeeze("band", drop=True)
dem_utm = dem.rio.reproject("EPSG:32645", resolution=30, resampling=Resampling.bilinear)
left, bottom, right, top = dem_utm.rio.bounds()
extent_km = [left / 1000, right / 1000, bottom / 1000, top / 1000]   # for maps, as in C4
print(dem_utm.shape, dem_utm.rio.crs)

In [ ]:
dem_ft = dem_utm * 3.28084      # 1 m = 3.28084 ft
print(type(dem_ft).__name__, dem_ft.rio.crs)
print("highest:", round(float(dem_utm.max()), 1), "m = about", round(float(dem_ft.max()), -1), "ft")

mean_m = float(dem_utm.mean())
relative_m = dem_utm - mean_m   # one number subtracted from every cell
print("mean:", round(mean_m, 1), "m")
print("relative elevation from", round(float(relative_m.min())), "to", round(float(relative_m.max())), "m")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6.5))   # fig: the whole figure; ax: the map area inside it (A10)
im = ax.imshow(relative_m.values, extent=extent_km, cmap="RdBu_r", vmin=-1500, vmax=1500)
fig.colorbar(im, ax=ax, label="Elevation minus the mean (m)", shrink=0.8)
ax.set_title("Above (red) and below (blue) the mean elevation")
ax.set_xlabel("Easting (km)")
ax.set_ylabel("Northing (km)")
plt.show()

`dem_ft` is still a DataArray in EPSG:32645. The highest cell, about 3,977.2 m, is about 13,050 ft. `round(value, -1)` rounds to the nearest ten: a negative number of decimals rounds to the left of the decimal point.

The mean of the grid is about 1,531.0 m. Relative elevation runs from about −1,006 m, in the deepest valleys, to about +2,446 m, on the highest ridge in the north-east.

On the map the Kathmandu Valley floor is pale blue: a little below the mean. The deep river valleys are dark blue and the northern mountains red. A diverging colour map with equal `vmin` and `vmax` puts white exactly at zero, so the colours mean "above" and "below".

*Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA).*

### Practice

**Q1 · one line:** Convert the whole grid to **kilometres** and print the highest value, rounded to 2 decimals.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
round(float((dem_utm / 1000).max()), 2)
```

3.98 km. Division works cell by cell like multiplication; `float()` turns xarray's one-value result into a plain number.
</details>

**Q2 · two lines:** What share of the cells with data lies **above** the mean? Count the cells of `relative_m` greater than 0, divide by the number of cells with data (`dem_utm.notnull().sum()`), and print it as a percentage, rounded to 1 decimal.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
above = int((relative_m > 0).sum())
print(round(above / int(dem_utm.notnull().sum()) * 100, 1))
```

About 41.7% of the cells lie above the mean. Fewer than half, because a smaller area of high mountains pulls the mean up, as you saw with the median in C1.
</details>

**Q3 · three lines:** A **standard score** says how many standard deviations a cell lies from the mean: (cell − mean) / standard deviation. Compute it for the whole grid with `float(dem_utm.std())`, then print its lowest and highest values, rounded to 2 decimals.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
std_m = float(dem_utm.std())
score = (dem_utm - mean_m) / std_m
print(round(float(score.min()), 2), round(float(score.max()), 2))
```

About −1.92 to +4.67. The highest ground lies more than four standard deviations above the mean, the lowest less than two below it: the DEM has a long tail of high mountains.
</details>

## 2. The integer overflow trap

The file stores elevations as `int16` (C1): whole numbers from −32,768 to 32,767. Arithmetic on an `int16` array gives an `int16` result.

If a result does not fit, it does not stop with an error. It **wraps around**, like the `uint8` example in A8: 32,767 + 1 becomes −32,768. This is **integer overflow**, and NumPy does it silently on whole arrays.

The example converts elevations to **decimetres** (1 m = 10 dm), as some programs do to keep one decimal in whole numbers. The highest cell, 3,979 m, should become 39,790 dm, which does not fit in `int16`.

The fix is always the same: convert to a float **before** calculating, with `.astype("float32")`. Grids opened with `masked=True` are already `float32`, which is one reason the course uses them.

**Predict:** what will the highest value in decimetres be?

In [ ]:
with rasterio.open(DEM_PATH) as src:
    raw_m = src.read(1)          # int16, as stored in the file

raw_dm = raw_m * 10              # still int16!
print(raw_dm.dtype)
print("highest:", raw_dm.max(), "dm | lowest:", raw_dm.min(), "dm")
print("cells with negative 'elevations':", (raw_dm < 0).sum())

In [ ]:
safe_dm = raw_m.astype("float32") * 10
print(safe_dm.dtype)
print("highest:", safe_dm.max(), "dm | lowest:", safe_dm.min(), "dm")

The `int16` result reports a highest value of 32,760 dm and a lowest of **−32,766 dm**, and 31,259 cells became negative. No error and no warning: only the impossible numbers give it away.

After `astype("float32")`, the highest cell is 39,790 dm and the lowest 5,250 dm, as they should be.

Remember the pattern from A8: **convert first, calculate second**. Integer grids are fine for storing whole metres, but unsafe for calculating with them.

### Practice

**Q1 · one line:** What is the largest elevation in metres that still fits in `int16` after multiplying by 10? Divide the largest `int16`, `np.iinfo(np.int16).max` (C1), by 10.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
np.iinfo(np.int16).max / 10
```

3,276.7 m. Every cell higher than that overflows when converted to decimetres in `int16`.
</details>

**Q2 · two lines:** Check that claim: count the cells of `raw_m` that are **3,277 m or higher**, and print the count next to the number of negative cells in `raw_dm`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
high_cells = (raw_m >= 3277).sum()
print(high_cells, (raw_dm < 0).sum())
```

31,259 and 31,259: exactly the cells above 3,276.7 m wrapped round to negative numbers. An overflow hits the highest values first, the ones a hazard study cares about most.
</details>

**Q3 · three lines:** Convert `raw_m` to **centimetres** safely, as `float64` this time, and print the dtype, the highest value and the mean, rounded to 1 decimal.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
raw_cm = raw_m.astype("float64") * 100
print(raw_cm.dtype, raw_cm.max())
print(round(raw_cm.mean(), 1))
```

`float64`, 397,900 cm and a mean of about 153,116.2 cm (1,531.2 m on the original grid). `float64` uses 8 bytes per cell instead of 4, so use `float32` when memory matters.
</details>

## 3. Comparisons, masks and areas

A comparison on a grid gives a grid of `True` and `False`: a **mask** (A8). `dem_utm > 2000` marks every cell above 2,000 m.

`True` counts as 1, so `.sum()` counts the cells, and **cells × 0.0009 km²** gives the area. That only works on a grid with equal cells in metres, which is why C4 built one.

Combine masks with `&` (and), `|` (or) and `~` (not), with brackets around each comparison, as in A9.

A trap: any comparison with `NaN` gives `False`. So `~(dem_utm > 2000)` is `True` for low ground **and** for the empty corners. To count "2,000 m or lower", ask for `dem_utm <= 2000` directly.

In [ ]:
high = dem_utm > 2000
print(type(high).__name__, high.dtype)
print("cells above 2,000 m:", int(high.sum()))
print("area above 2,000 m:", round(int(high.sum()) * CELL_KM2, 1), "km²")

middle = (dem_utm >= 1000) & (dem_utm < 2000)
print("area from 1,000 to 2,000 m:", round(int(middle.sum()) * CELL_KM2), "km²")

print("not high:", int((~high).sum()), "| 2,000 m or lower:", int((dem_utm <= 2000).sum()))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6.5))
ax.imshow(middle.values, extent=extent_km, cmap="Greens")
ax.set_title("Cells from 1,000 to 2,000 m (green)")
ax.set_xlabel("Easting (km)")
ax.set_ylabel("Northing (km)")
plt.show()

`high` is a DataArray of type `bool`. About 517,100 cells lie above 2,000 m: about 465.4 km². The middle hills from 1,000 to 2,000 m cover about 2,082 km², most of the grid.

`~high` counts about 2,841,700 cells but `dem_utm <= 2000` only about 2,758,900. The difference, 82,774, is exactly the number of `NaN` cells in the corners (C4): `~` turned their `False` into `True`.

A mask map needs no colour bar: green is `True`, white is `False`. The deep river valleys (below 1,000 m) and the high ground (above 2,000 m) stay white, and so do the empty corners.

*Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA).*

### Practice

**Q1 · one line:** How many km² of the grid lie **below 1,400 m**? Round to 0 decimals.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
round(int((dem_utm < 1400).sum()) * CELL_KM2)
```

About 1,384 km²: almost half the area, mostly the Kathmandu Valley floor and the river valleys.
</details>

**Q2 · two lines:** Make a mask of the band from **1,500 to 2,500 m** with `&`, and print its area in km², rounded to 1 decimal.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
band = (dem_utm >= 1500) & (dem_utm < 2500)
print(round(int(band.sum()) * CELL_KM2, 1))
```

About 1,148 km² (the decimal can differ slightly between versions). Each comparison sits in its own brackets, because `&` is evaluated before `>=` and `<`.
</details>

**Q3 · three lines:** What share of the area with data is **either** below 800 m **or** above 3,000 m? Combine two masks with `|`, then print the area in km² and the percentage of all cells with data, each rounded to 1 decimal.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
extreme = (dem_utm < 800) | (dem_utm > 3000)
extreme_km2 = int(extreme.sum()) * CELL_KM2
print(round(extreme_km2, 1), round(extreme_km2 / (int(dem_utm.notnull().sum()) * CELL_KM2) * 100, 1))
```

About 190.3 km², roughly 6.5% of the area: the deepest gorges and the highest ridges together are a small part of the landscape.
</details>

## 4. Choosing values cell by cell: `np.where` and `xr.where`

`np.where(condition, a, b)` from A8 builds a new grid: where the condition is `True` it takes `a`, elsewhere `b`. `a` and `b` can be single numbers or whole grids.

`xr.where(condition, a, b)` does the same for DataArrays and **keeps the coordinates and the CRS**. `np.where` returns a plain NumPy array with no map information.

A common use: keep the values you want and turn the rest into `NaN`, so that later statistics only see the cells you chose.

Another: cap values, for example to show everything above 3,000 m as 3,000 m on a map.

In [ ]:
high_only = xr.where(dem_utm > 2000, dem_utm, np.nan)
print(type(high_only).__name__, high_only.rio.crs)
print("mean of the ground above 2,000 m:", round(float(high_only.mean()), 1), "m")

dem_m = dem_utm.values                        # the plain NumPy array behind the grid
capped = np.where(dem_m > 3000, 3000, dem_m)  # NaN > 3000 is False, so NaN stays NaN
print(type(capped).__name__, np.nanmax(capped))

`xr.where` returned a DataArray that still knows its CRS, EPSG:32645. The ground above 2,000 m has a mean of about 2,449.0 m.

`dem_utm.values` is the NumPy array inside the DataArray. `np.where` on it returns a NumPy `ndarray`, with every value above 3,000 m replaced by 3,000.0. The `NaN` cells stayed `NaN`, because `NaN > 3000` is `False` and the third argument, `dem_m`, is `NaN` there.

`np.nanmax` is `max` that skips `NaN`; step 7 explains why that matters.

### Practice

**Q1 · one line:** What is the mean elevation of the ground **below 1,400 m**? Use `xr.where` and round to 1 decimal.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
round(float(xr.where(dem_utm < 1400, dem_utm, np.nan).mean()), 1)
```

About 1,125.9 m (the decimal can differ slightly between versions). It mixes the valley floor at about 1,300 m with the much lower river valleys.
</details>

**Q2 · two lines:** Do the same with `np.where` on `dem_m`, and take the mean with `np.nanmean`. Print it rounded to 1 decimal.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
low_only = np.where(dem_m < 1400, dem_m, np.nan)
print(round(float(np.nanmean(low_only)), 1))
```

The same value as Q1. Both functions choose cell by cell; `xr.where` keeps the map information, `np.where` gives a bare array.
</details>

**Q3 · three lines:** Make a grid that is 1 where the ground is **below 1,300 m** and 0 elsewhere, but keeps `NaN` where there is no data. Use two `np.where` calls, one after the other: the second uses `np.isnan(dem_m)` to put `NaN` back into the empty cells. Print the distinct values with `np.unique`.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
low_flag = np.where(dem_m < 1300, 1, 0)
low_flag = np.where(np.isnan(dem_m), np.nan, low_flag)
print(np.unique(low_flag))
```

`[ 0.  1. nan]`. `np.isnan` is `True` where a value is `NaN`. Without the second line, the empty corners would read 0, "not low", which claims knowledge you do not have.
</details>

## 5. Reclassifying elevation into zones

To **reclassify** a raster is to replace ranges of values with class codes. Here the DEM becomes five elevation zones. These are **made-up teaching classes**, not an official classification.

`np.select(conditions, codes, default=0)` takes a list of masks and a list of codes. Each cell gets the code of the **first** condition that is `True`, or the default if none is. Here the default 0 marks the empty cells, because every comparison with `NaN` is `False`.

`np.digitize(values, edges)` is a shorter alternative: it numbers the bins between the edges 0, 1, 2… But it sorts `NaN` into the **last** bin, as if empty cells were mountains. One of the questions shows this.

Zone codes are categories (C4): to resample them, use `nearest` or `mode`.

In [ ]:
conditions = [
    dem_m < 1000,
    (dem_m >= 1000) & (dem_m < 1500),
    (dem_m >= 1500) & (dem_m < 2000),
    (dem_m >= 2000) & (dem_m < 3000),
    dem_m >= 3000,
]
zone_codes = [1, 2, 3, 4, 5]
zone_names = {1: "below 1,000 m", 2: "1,000-1,500 m", 3: "1,500-2,000 m", 4: "2,000-3,000 m", 5: "3,000 m and above"}

zones = np.select(conditions, zone_codes, default=0)   # 0 = no data
print(zones.dtype, zones.shape)
print(np.unique(zones))

In [ ]:
zones_map = np.where(zones == 0, np.nan, zones)   # 0 (no data) left blank on the map

fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(zones_map, extent=extent_km, cmap=plt.get_cmap("viridis", 5), vmin=0.5, vmax=5.5)
cbar = fig.colorbar(im, ax=ax, shrink=0.8, ticks=zone_codes)
cbar.ax.set_yticklabels([zone_names[code] for code in zone_codes])
ax.set_title("Elevation zones (made-up teaching classes)")
ax.set_xlabel("Easting (km)")
ax.set_ylabel("Northing (km)")
plt.show()

`zones` is a whole-number array with the codes 0 to 5. Each condition sits in brackets, and `&` joins the two ends of each range. The ranges include their lower limit and exclude their upper one, so no cell falls into two zones.

`plt.get_cmap("viridis", 5)` cuts the colour map into five colours, from dark purple (low) to yellow (high), and `vmin=0.5, vmax=5.5` centres each code on its colour. The colour bar shows names instead of numbers, using the dictionary `zone_names` (A4).

The map simplifies the DEM into five readable areas: the valley floor in zone 2, the hills around it in zones 3 and 4, and the high ground of zone 5 in the north.

*Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA).*

### Practice

**Q1 · one line:** Look up the name of zone 4 in `zone_names`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
zone_names[4]
```

'2,000-3,000 m'. Keeping codes and names in a dictionary keeps maps, tables and legends consistent.
</details>

**Q2 · two lines:** Reclassify with `np.digitize(dem_m, [1000, 1500, 2000, 3000])` instead, and print the distinct values with their counts, using `np.unique(..., return_counts=True)`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
bins = np.digitize(dem_m, [1000, 1500, 2000, 3000])
print(np.unique(bins, return_counts=True))
```

The bins are numbered 0 to 4, and bin 4 holds about 150,000 cells: the 67,000 or so cells above 3,000 m **plus** the 82,774 `NaN` cells. `np.digitize` hid the empty corners among the mountains. Mask `NaN` first, or use `np.select`.
</details>

**Q3 · three lines:** Make your own **three** teaching zones with `np.select`: 1 below 1,500 m, 2 from 1,500 to 2,500 m, 3 from 2,500 m up, and 0 for no data. Print the distinct codes and their counts.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
my_conditions = [dem_m < 1500, (dem_m >= 1500) & (dem_m < 2500), dem_m >= 2500]
my_zones = np.select(my_conditions, [1, 2, 3], default=0)
print(np.unique(my_zones, return_counts=True))
```

Four codes: 0 (82,774 empty cells), 1 (about 1.83 million cells), 2 (about 1.28 million) and 3 (about 0.17 million). Changing the edges changes every map and every area: class edges are a choice, so always state them.
</details>

## 6. Counting cells and area per zone

A reclassified raster answers questions such as "how much land lies in each zone?" Count the cells of each code, then multiply by the cell area.

`np.unique(array, return_counts=True)` returns two arrays: the distinct values and how many times each occurs. Leave out code 0 first with a mask, `zones[zones > 0]`, which keeps only the cells with data as one long row of values.

Put the result into a pandas table (A9) to add areas, percentages and names. This kind of table appears in most hazard and land-use reports.

In [ ]:
codes, counts = np.unique(zones[zones > 0], return_counts=True)

zone_table = pd.DataFrame({"zone": codes, "cells": counts})
zone_table["name"] = zone_table["zone"].map(zone_names)
zone_table["area_km2"] = zone_table["cells"] * CELL_KM2
zone_table["percent"] = zone_table["cells"] / zone_table["cells"].sum() * 100
zone_table.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(zone_table["name"], zone_table["area_km2"], color="peru")
ax.set_xlabel("Area (km²)")
ax.set_title("Area of each elevation zone (made-up teaching classes)")
plt.show()

Zone 2 (1,000 to 1,500 m) is the largest, about 1,242.4 km² or 42.1% of the area with data. Zone 3 follows with about 839.1 km² (28.5%). Zones 1 and 4 cover about 401.5 and 405 km², and zone 5, the highest ground, only about 60.4 km² (2.0%).

`.map(zone_names)` looks up each code in the dictionary and adds its name, a pandas tool for translating codes. The bar chart starts at zero and has units, as A10's checklist asks.

### Practice

**Q1 · one line:** What is the total area of all five zones? Sum the `area_km2` column and round to 1 decimal.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
round(zone_table["area_km2"].sum(), 1)
```

About 2,948.4 km², the same as the area with data in C4. Every cell with data fell into exactly one zone: a good check of the conditions.
</details>

**Q2 · two lines:** How much land lies at **2,000 m or more** (zones 4 and 5 together)? Filter the table and sum the areas, rounded to 1 decimal.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
upper = zone_table[zone_table["zone"] >= 4]
print(round(upper["area_km2"].sum(), 1))
```

About 465.4 km², the same as the "above 2,000 m" mask in step 3. Two different routes, one answer: that is how you know both are right.
</details>

**Q3 · three lines:** Add a column `mean_m` with the mean elevation of each zone. Loop over `zone_table["zone"]`, take `float(dem_m[zones == code].mean())` for each code, and store the list as the new column.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
means = []
for code in zone_table["zone"]:
    means.append(float(dem_m[zones == code].mean()))
zone_table["mean_m"] = means
zone_table.round(1)
```

The means rise from about 838.5 m in zone 1 to about 3,275 m in zone 5, each inside its own range. `float()` turns NumPy's `float32` result into a plain Python number, so `.round(1)` prints it neatly. `dem_m[zones == code]` uses one grid as a mask for another: both have the same shape, cell for cell.
</details>

## 7. Nodata in calculations: `NaN` and masked arrays

Missing cells must stay out of every statistic. There are two common ways to mark them.

**`NaN`** ("not a number") is a float value. Any arithmetic with `NaN` gives `NaN`, so `np.mean` and `np.max` on a grid with one `NaN` return `nan`. The `nan` functions, `np.nanmean`, `np.nanmax` and `np.nanmin`, skip it. xarray's own `.mean()` and `.max()` skip `NaN` for you, which is why earlier steps worked.

`np.isfinite(grid)` is `True` for real numbers. `grid[np.isfinite(grid)]` keeps only the valid cells, as one long 1-D array. `np.where(valid, values, np.nan)` blanks cells you have no data for.

A **masked array** keeps the original values plus a separate True/False mask of cells to ignore. `src.read(1, masked=True)` returns one, and `np.ma.masked_equal(array, nodata)` makes one. Its `.mean()` ignores the masked cells. This suits integer grids, which cannot hold `NaN`.

In [ ]:
print("np.mean:", np.mean(dem_m), "| np.nanmean:", round(float(np.nanmean(dem_m)), 1))
print("np.max:", np.max(dem_m), "| np.nanmax:", round(float(np.nanmax(dem_m)), 1))

valid = np.isfinite(dem_m)
values = dem_m[valid]                 # the valid cells, as one long 1-D array
print(valid.sum(), values.shape, round(float(values.mean()), 1))

In [ ]:
# A hole made ON PURPOSE in the int16 grid from step 2
holed_m = raw_m.copy()
holed_m[300:400, 300:400] = -32768

print("plain mean:", round(holed_m.mean(), 1))
masked_m = np.ma.masked_equal(holed_m, -32768)
print("masked mean:", round(masked_m.mean(), 1), "| masked cells:", masked_m.mask.sum())
print("original mean:", round(raw_m.mean(), 1))

`np.mean` and `np.max` return `nan`: 82,774 empty cells poison the whole result. `np.nanmean` and `np.nanmax` give about 1,531.0 m and 3,977.2 m.

`valid` counts 3,276,043 cells with data, and `values` holds them in a 1-D array of the same length.

The 100 × 100 hole is only 10,000 cells, but the plain mean drops to about 1,434.7 m. The masked array ignores those 10,000 cells and gives about 1,532.7 m, close to the original 1,531.2 m. The small difference is real: the masked mean has lost the hole's own elevations.

### Practice

**Q1 · one line:** What is the lowest elevation in the UTM grid? Use the right `nan` function on `dem_m`, rounded to 1 decimal.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
round(float(np.nanmin(dem_m)), 1)
```

About 525.3 m. Plain `np.min` would give `nan`.
</details>

**Q2 · two lines:** Using `values` (the valid cells), print the 10th and 90th percentiles with `np.percentile(values, [10, 90])`, rounded to 0 decimals.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
low_m, high_m = np.percentile(values, [10, 90])
print(round(float(low_m)), round(float(high_m)))
```

About 927 m and about 2,208 m: 80% of the ground lies between them. Percentile functions do not skip `NaN`, so pass them the valid values only.
</details>

**Q3 · three lines:** Read the DEM file with `src.read(1, masked=True)` into `dem_masked`, then print its type and its mean, rounded to 1 decimal.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
with rasterio.open(DEM_PATH) as src:
    dem_masked = src.read(1, masked=True)
print(type(dem_masked).__name__, round(dem_masked.mean(), 1))
```

`MaskedArray` and about 1,531.2 m. The file has no nodata cells, so nothing is masked; in a file with holes, rasterio would mask every −32768 for you.
</details>

## 8. Filling holes with `fillnodata`

Sometimes a DEM has **voids**: cells with no value, under cloud or in deep shadow. For some calculations you need a value everywhere, so you **fill** the holes with a best guess.

`rasterio.fill.fillnodata(array, mask=valid, max_search_distance=100)` fills each missing cell from the valid cells around it, searching up to 100 cells away. In `mask`, `True` means "valid, keep" and `False` means "fill me".

A filled value is an **estimate**, not a measurement. The example tests how good it is. It works on the original grid in degrees (the file has no empty corners), cuts two 20 × 20 holes **on purpose**, one on the flat valley floor at Kathmandu Durbar Square and one on steep ground in the north-east, fills them, and compares the guesses with the true values it kept.

In [ ]:
from rasterio.fill import fillnodata

truth_m = raw_m.astype("float32")          # the original grid in degrees, as float
holes = {"valley floor": (1056, 375), "steep ground": (200, 1800)}   # top-left row, col of each hole

valid_cells = np.ones(truth_m.shape, dtype=bool)
for row, col in holes.values():
    valid_cells[row:row + 20, col:col + 20] = False     # holes made ON PURPOSE

holed = np.where(valid_cells, truth_m, np.nan)
print("empty cells before filling:", np.isnan(holed).sum())
filled = fillnodata(holed, mask=valid_cells, max_search_distance=100)
print("empty cells after filling:", np.isnan(filled).sum())

for name, (row, col) in holes.items():
    error_m = filled[row:row + 20, col:col + 20] - truth_m[row:row + 20, col:col + 20]
    print(f"{name:13} mean error {np.abs(error_m).mean():5.1f} m, largest {np.abs(error_m).max():6.1f} m")

Before filling, 800 cells were empty (two holes of 400); afterwards, none. `fillnodata` may write into the array you give it, so count the holes **before** filling, and keep the truth in a separate array, as `truth_m` here.

On the valley floor the filled values are off by about 3.6 m on average, and at most about 12.6 m: on flat ground a guess from the neighbours is good. On the steep ground the average error is more than ten times larger, about 42.9 m, and single cells are off by over 100 m. `fillnodata` spreads the edge values smoothly inwards; it cannot invent a ridge or a gully that it never saw.

For hazard work this matters: a filled void on a steep slope can hide a scarp or a gully. Keep a record of which cells were filled, for example the `valid_cells` mask, and treat them with care.

### Practice

**Q1 · one line:** How many cells did the two holes cover in total? Count the `False` values in `valid_cells` with `~`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
(~valid_cells).sum()
```

800 cells: two holes of 20 × 20. `~` turns each `False` into `True`, so `.sum()` counts the holes.
</details>

**Q2 · two lines:** Print the true elevations of the top-left 3 × 3 cells of the steep hole, and the filled values of the same cells, rounded to 0 decimals with `np.round` (A8).

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(np.round(truth_m[200:203, 1800:1803]))
print(np.round(filled[200:203, 1800:1803]))
```

Along the edge of the hole (the first row and the first column), the filled values are within about 16 m of the truth, because `fillnodata` found real neighbours there. Two cells further in, the error is already about 60 m: 2,211 m filled against 2,152 m true. The errors grow towards the centre of the hole.
</details>

**Q3 · three lines:** Fill again with a **short** search distance, `max_search_distance=5`, and count how many cells stay empty. Use a fresh copy: `np.where(valid_cells, truth_m, np.nan)`.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
holed_again = np.where(valid_cells, truth_m, np.nan)
short = fillnodata(holed_again, mask=valid_cells, max_search_distance=5)
print(np.isnan(short).sum())
```

200 cells stay empty: the 10 × 10 centre of each hole lies more than 5 cells from any valid cell, so nothing was found to fill it. The search distance limits how far a guess may reach.
</details>

## 9. When it breaks: `and` with two grids

You want the cells between 1,000 and 2,000 m, and you write the condition in plain English with `and`. Run the cell.

In [ ]:
middle_hills = (dem_utm >= 1000) and (dem_utm < 2000)

The last line of the error says:

`ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()`

- **truth value** means a single `True` or `False`.
- Python's `and` needs one truth value on each side, as in A4's `if` statements. A grid of 3.3 million True/False values is not one value: it is **ambiguous**.
- The hint, `a.any()` or `a.all()`, would squash the grid into one value. That is not what you want here.

The fix: use `&` (cell-by-cell "and") instead of `and`, with each comparison in brackets: `(dem_utm >= 1000) & (dem_utm < 2000)`, as in step 3. Likewise use `|` for `or` and `~` for `not`.

**Try it:** fix the cell.

## Mixed review

These exercises combine everything in this lesson. Each one needs 5 to 10 lines. Plan in words first, then write the code.

### Review 1 · A zone report saved as CSV

Rebuild the zone table from steps 5 and 6 in one cell, with the columns `zone`, `name`, `area_km2`, `percent` and `mean_m`. Sort it by area, largest first, save it as `c5_elevation_zones.csv` in `OUT` (A9) and print it rounded to 1 decimal.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
codes, counts = np.unique(zones[zones > 0], return_counts=True)
report = pd.DataFrame({"zone": codes, "name": [zone_names[c] for c in codes]})
report["area_km2"] = counts * CELL_KM2
report["percent"] = counts / counts.sum() * 100
report["mean_m"] = [float(dem_m[zones == c].mean()) for c in codes]

report = report.sort_values("area_km2", ascending=False)
report.to_csv(OUT / "c5_elevation_zones.csv", index=False)
print(report.round(1))
```

Zone 2 heads the table with about 1,242.4 km², zone 5 closes it with about 60.4 km². The list comprehensions (A3) build the names and means in one line each. The CSV can go straight into a report or a spreadsheet.
</details>

### Review 2 · Is the north higher?

xarray can compare a grid with its own coordinates: `dem_utm.y` holds the northing of each row. Split the grid at its middle northing, `(top + bottom) / 2`. For the northern and the southern half, print the area above 2,500 m in km², rounded to 1 decimal.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
middle_y = (top + bottom) / 2
north_half = dem_utm.y > middle_y
above_2500 = dem_utm > 2500

north_km2 = int((above_2500 & north_half).sum()) * CELL_KM2
south_km2 = int((above_2500 & ~north_half).sum()) * CELL_KM2
print("north:", round(north_km2, 1), "km² | south:", round(south_km2, 1), "km²")
```

Almost all the ground above 2,500 m lies in the northern half: about 154.7 km² against about 1.8 km² in the south. `dem_utm.y > middle_y` is a mask of one value per row; combined with the 2-D grid, xarray stretches it across every column (broadcasting). The `~` is safe here, because the coordinates have no `NaN`.
</details>

### Review 3 · Save the zones as a GeoTIFF

Write `zones` to `c5_elevation_zones.tif` in `OUT` with rasterio, as in C1: one band, `dtype="uint8"` (codes 0 to 5 fit in 0 to 255), the CRS and transform of `dem_utm`, and `nodata=0`. Read it back and print the dtype, the nodata value and the distinct codes.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
zones_path = OUT / "c5_elevation_zones.tif"
with rasterio.open(zones_path, "w", driver="GTiff", height=zones.shape[0], width=zones.shape[1],
                   count=1, dtype="uint8", crs=dem_utm.rio.crs,
                   transform=dem_utm.rio.transform(), nodata=0) as dst:
    dst.write(zones.astype("uint8"), 1)

with rasterio.open(zones_path) as src:
    back = src.read(1)
    print(src.dtypes, src.nodata, np.unique(back))
```

`uint8`, nodata 0 and the codes 0 to 5. `zones` was `int64`, 8 bytes per cell; `uint8` takes 1 byte, so the codes take an eighth of the space. `nodata=0` tells every later program that 0 means no data, the lesson of C4's mosaics.
</details>

### Review 4 · Bigger holes, bigger errors

Repeat the test of step 8 for **one** square hole on the steep ground at row 200, column 1800, with sides of 10, 20 and 40 cells. For each size print the mean and the largest absolute error in m, rounded to 1 decimal.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
for size in [10, 20, 40]:
    keep = np.ones(truth_m.shape, dtype=bool)
    keep[200:200 + size, 1800:1800 + size] = False
    guess = fillnodata(np.where(keep, truth_m, np.nan), mask=keep, max_search_distance=100)
    error_m = np.abs(guess[200:200 + size, 1800:1800 + size] - truth_m[200:200 + size, 1800:1800 + size])
    print(size, round(float(error_m.mean()), 1), round(float(error_m.max()), 1))
```

The mean error grows from about 27.7 m (10 cells) to about 45.8 m (40 cells), and the largest from about 77 m to about 147 m. The larger the hole, the more of the real shape lies far from any known cell. Small voids can be filled with little harm; large voids on steep ground need another data source.
</details>

## Recap

- **Map algebra**: arithmetic, comparisons and functions work on every cell at once. xarray results keep their coordinates and CRS.
- Integer grids **overflow silently** (`int16` wraps above 32,767). Convert with `.astype("float32")` before calculating.
- A comparison gives a **mask**; `.sum()` counts its cells and cells × 0.0009 km² gives the area on the 30 m UTM grid. Combine masks with `&`, `|`, `~` (never `and`/`or`), and remember that `NaN` compares as `False`.
- `np.where` / `xr.where` choose values cell by cell; `np.select` **reclassifies** ranges into codes (watch `np.digitize`, which puts `NaN` in the last bin). `np.unique(..., return_counts=True)` gives cells and areas per class.
- Nodata must stay out of statistics: use `nan` functions or `np.isfinite`, masked arrays for integer grids, and fill voids with `fillnodata` only as a marked, tested estimate.

### Words you learned

| Word | Meaning |
| --- | --- |
| map algebra / raster algebra | Calculating with whole rasters, cell by cell |
| relative elevation | A cell's elevation minus a reference, here the mean of the grid |
| standard score | (value − mean) / standard deviation: how unusual a value is |
| integer overflow | A result too large for its integer type, which wraps round to a wrong value |
| mask | A grid of True/False values marking the cells that meet a condition |
| `xr.where` | Chooses values cell by cell, like `np.where`, and keeps coordinates and CRS |
| reclassify | Replace ranges of values with class codes |
| `np.select` | Gives each cell the code of the first condition that is True |
| `np.digitize` | Numbers the bins between given edges; it puts NaN in the last bin |
| teaching classes | Classes made up for practice, not an official scheme |
| `NaN` | "Not a number": the float value that marks missing data |
| `np.nanmean` / `np.nanmax` | Mean and maximum that skip NaN |
| `np.isfinite` | True where a value is a real number (not NaN or infinite) |
| masked array | A NumPy array plus a mask of cells to ignore in calculations |
| void | A hole in a DEM: cells with no value |
| `fillnodata` | rasterio function that fills missing cells from the valid cells around them |

## Next

**C6 · Terrain analysis: slope, aspect and hillshade:** turn the DEM into the slope, aspect and shaded-relief maps at the heart of every landslide study.

<!-- pygile-nav -->
**PyGILE Course** · lesson 29 of 39 · [Course home](../README.md)  
← Previous: [C4 · Reproject, resample and mosaic: the DEM on a grid in metres](C04_Reproject_resample_and_mosaic.ipynb) · Next: [C6 · Terrain analysis: slope, aspect and hillshade for landslide studies](C06_Terrain_analysis.ipynb) →